
# RSNA Knee Abnormality Detection — Label Audit + Research Manifest v1

## Mục tiêu

Notebook này là **bước kế tiếp sau EDA**, chưa train MRI model.

Nó làm 3 việc:

1. **Audit các nguồn pseudo/LLM labels**
   - Tự tìm các file label trong `/kaggle/input`
   - Chuẩn hóa tên cột
   - Join theo `StudyInstanceUID`
   - So với 58 gold labels theo từng pathology
   - Đánh giá AUC / AP / accuracy / missingness / disagreement
   - Tạo vài candidate silver-label files để dùng cho baseline sau

2. **Giữ External Data Manifest**
   - Dataset nào đáng nghiên cứu
   - Hữu ích cho pathology nào
   - Dùng để classification / localization / pretraining
   - Trạng thái license/access: `USE / HOLD / REJECT / VERIFY`

3. **Giữ Knee MRI Domain-Knowledge Map**
   - 12 pathology
   - vùng giải phẫu
   - plane/sequence ưu tiên
   - modeling hook cần test sau này
   - phân biệt `evidence-backed` và `hypothesis-to-test`

> **Nguyên tắc:** 58 gold image labels là referee. Report-derived labels chỉ là weak/silver supervision.

---

## Trước khi chạy trên Kaggle

Attach:
- Competition input: **RSNA Knee Abnormality Detection**
- Các public label datasets bạn muốn audit, ví dụ:
  - `RSNA Knee LLM-read report labels`
  - các LLM/pseudo-label datasets khác bạn tìm được

Notebook sẽ cố gắng tự tìm CSV/Parquet/JSON có `StudyInstanceUID` + các label tương ứng.

**GPU: không cần.**


In [ ]:

# ============================================================
# 0. Imports & config
# ============================================================
from pathlib import Path
from collections import defaultdict
import os, re, json, math, hashlib, warnings, zipfile, statistics

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score
)

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

FAST_MODE = True
BOOTSTRAP_ITERS = 500 if FAST_MODE else 3000

LABELS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

print("FAST_MODE:", FAST_MODE)
print("BOOTSTRAP_ITERS:", BOOTSTRAP_ITERS)


In [ ]:

# ============================================================
# 1. Find competition root robustly
# ============================================================
def find_competition_root():
    roots = []
    base = Path("/kaggle/input")

    for train_csv in base.rglob("train.csv"):
        try:
            cols = pd.read_csv(train_csv, nrows=2).columns.tolist()
            if "StudyInstanceUID" in cols and "ACL" in cols and "MCL" in cols:
                roots.append(train_csv.parent)
        except Exception:
            pass

    if not roots:
        raise FileNotFoundError(
            "Could not locate RSNA Knee competition train.csv. "
            "Please attach the competition input."
        )

    preferred = [
        r for r in roots
        if "knee" in str(r).lower() or "rsna" in str(r).lower()
    ]

    root = preferred[0] if preferred else roots[0]
    return root

COMP_ROOT = find_competition_root()
TRAIN_CSV = COMP_ROOT / "train.csv"

print("Competition root:", COMP_ROOT)
print("train.csv:", TRAIN_CSV)


In [ ]:

# ============================================================
# 2. Load competition train and identify 58 gold studies
# ============================================================
train = pd.read_csv(TRAIN_CSV)

missing_expected = [c for c in LABELS if c not in train.columns]
assert not missing_expected, f"Missing competition labels: {missing_expected}"

train["n_available_labels"] = train[LABELS].notna().sum(axis=1)
gold_mask = train["n_available_labels"].eq(len(LABELS))

gold = train.loc[gold_mask, ["StudyInstanceUID"] + LABELS].copy()
weak_pool = train.loc[~gold_mask, ["StudyInstanceUID", "Report"]].copy()

print("Train studies:", len(train))
print("Gold studies:", len(gold))
print("Weak/unstructured studies:", len(weak_pool))
display(gold.head())



## 3. Tự động tìm pseudo-label sources

Notebook không mặc định schema của dataset public.

Nó thử tìm:
- `.csv`
- `.parquet`
- `.json`

sau đó kiểm tra:
- có UID giống `StudyInstanceUID`
- có ít nhất một cột có thể map về 12 targets

Nếu một file bị bỏ sót, cuối section này có cell để **manual register**.


In [ ]:

# ============================================================
# 3. Label-name normalization / alias mapping
# ============================================================
def norm_name(s):
    s = str(s).strip().lower()
    s = s.replace("'", "")
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

ALIASES = {
    "ACL": [
        "acl", "anterior cruciate ligament", "acl injury", "acl tear"
    ],
    "MCL": [
        "mcl", "medial collateral ligament", "mcl injury", "mcl tear"
    ],
    "Medial Meniscus": [
        "medial meniscus", "medial meniscal tear", "medial meniscus tear",
        "mm", "medial meniscal"
    ],
    "Lateral Meniscus": [
        "lateral meniscus", "lateral meniscal tear", "lateral meniscus tear",
        "lm", "lateral meniscal"
    ],
    "Medial OA": [
        "medial oa", "medial osteoarthritis",
        "medial tibiofemoral oa", "medial tibiofemoral osteoarthritis"
    ],
    "Lateral OA": [
        "lateral oa", "lateral osteoarthritis",
        "lateral tibiofemoral oa", "lateral tibiofemoral osteoarthritis"
    ],
    "PF OA": [
        "pf oa", "patellofemoral oa", "patellofemoral osteoarthritis",
        "patello femoral oa"
    ],
    "Effusion": [
        "effusion", "joint effusion"
    ],
    "Synovitis": [
        "synovitis"
    ],
    "Baker's": [
        "bakers", "baker cyst", "bakers cyst", "baker s cyst",
        "popliteal cyst"
    ],
    "Contusion": [
        "contusion", "bone contusion", "bone bruise", "bone bruising"
    ],
    "Fracture": [
        "fracture", "bone fracture"
    ],
}

ALIAS_TO_LABEL = {}
for label, aliases in ALIASES.items():
    ALIAS_TO_LABEL[norm_name(label)] = label
    for a in aliases:
        ALIAS_TO_LABEL[norm_name(a)] = label

UID_ALIASES = {
    "studyinstanceuid", "study instance uid", "study_uid", "study uid",
    "studyid", "study id", "uid"
}

def infer_uid_col(df):
    exact = {norm_name(c): c for c in df.columns}
    for k in UID_ALIASES:
        nk = norm_name(k)
        if nk in exact:
            return exact[nk]
    return None

def infer_label_cols(df):
    found = {}
    nmap = {norm_name(c): c for c in df.columns}

    # Exact alias matches first
    for nc, orig in nmap.items():
        if nc in ALIAS_TO_LABEL:
            found[ALIAS_TO_LABEL[nc]] = orig

    # Conservative fallback: allow common suffix/prefix patterns
    for label in LABELS:
        if label in found:
            continue
        target_norm = norm_name(label)
        candidates = []
        for c in df.columns:
            nc = norm_name(c)
            if (
                nc == target_norm
                or nc == target_norm + " score"
                or nc == target_norm + " probability"
                or nc == target_norm + " prob"
                or nc == target_norm + " label"
                or nc == target_norm + " pred"
                or nc == "pred " + target_norm
            ):
                candidates.append(c)
        if candidates:
            found[label] = candidates[0]

    return found


In [ ]:

# ============================================================
# 4. Recursively discover candidate label files
# ============================================================
EXCLUDE_NAMES = {
    "train.csv", "test.csv", "train_series.csv", "test_series.csv",
    "sample_submission.csv", "submission.csv"
}

def safe_read_table(path):
    path = Path(path)
    try:
        if path.suffix.lower() == ".csv":
            return pd.read_csv(path)
        elif path.suffix.lower() in {".parquet", ".pq"}:
            return pd.read_parquet(path)
        elif path.suffix.lower() == ".json":
            # Try normal JSON first, then JSON lines
            try:
                return pd.read_json(path)
            except Exception:
                return pd.read_json(path, lines=True)
    except Exception:
        return None
    return None

candidate_paths = []
for ext in ["*.csv", "*.parquet", "*.pq", "*.json"]:
    candidate_paths.extend(Path("/kaggle/input").rglob(ext))

rows = []

for p in candidate_paths:
    if p.name.lower() in EXCLUDE_NAMES:
        continue

    df = safe_read_table(p)
    if df is None or len(df) == 0:
        continue

    uid_col = infer_uid_col(df)
    label_cols = infer_label_cols(df)

    if uid_col is None or len(label_cols) == 0:
        continue

    overlap_n = len(set(df[uid_col].astype(str)) & set(train["StudyInstanceUID"].astype(str)))

    if overlap_n == 0:
        continue

    rows.append({
        "path": str(p),
        "source_name": p.parent.name + " / " + p.name,
        "n_rows": len(df),
        "uid_col": uid_col,
        "n_mapped_labels": len(label_cols),
        "mapped_labels": ", ".join(label_cols.keys()),
        "competition_uid_overlap": overlap_n,
    })

discovery = pd.DataFrame(rows).sort_values(
    ["n_mapped_labels", "competition_uid_overlap"],
    ascending=False
) if rows else pd.DataFrame()

print("Candidate pseudo-label sources found:", len(discovery))
display(discovery)


In [ ]:

# ============================================================
# 5. OPTIONAL manual source registration
#
# If auto-discovery misses a file, add path(s) here.
# Leave empty otherwise.
# ============================================================
MANUAL_LABEL_FILES = [
    # Example:
    # "/kaggle/input/some-label-dataset/labels.csv",
]

manual_rows = []
for p in MANUAL_LABEL_FILES:
    df = safe_read_table(p)
    if df is None:
        print("Could not read:", p)
        continue

    uid_col = infer_uid_col(df)
    label_cols = infer_label_cols(df)

    print("\nManual:", p)
    print("UID:", uid_col)
    print("Mapped:", label_cols)

    if uid_col and label_cols:
        manual_rows.append({
            "path": str(p),
            "source_name": Path(p).parent.name + " / " + Path(p).name,
            "n_rows": len(df),
            "uid_col": uid_col,
            "n_mapped_labels": len(label_cols),
            "mapped_labels": ", ".join(label_cols.keys()),
            "competition_uid_overlap": len(
                set(df[uid_col].astype(str))
                & set(train["StudyInstanceUID"].astype(str))
            ),
        })

if manual_rows:
    discovery = pd.concat([discovery, pd.DataFrame(manual_rows)], ignore_index=True)
    discovery = discovery.drop_duplicates("path")
    display(discovery)


In [ ]:

# ============================================================
# 6. Normalize all discovered sources into a common schema
# ============================================================
def coerce_score(s):
    # Handle booleans / numeric / textual states
    if pd.api.types.is_bool_dtype(s):
        return s.astype(float)

    numeric = pd.to_numeric(s, errors="coerce")
    numeric_rate = numeric.notna().mean()

    if numeric_rate >= 0.80:
        return numeric.astype(float)

    mapping = {
        "positive": 1.0, "pos": 1.0, "present": 1.0, "yes": 1.0, "true": 1.0,
        "negative": 0.0, "neg": 0.0, "absent": 0.0, "no": 0.0, "false": 0.0,
        "uncertain": 0.5, "indeterminate": 0.5, "possible": 0.5,
        "unknown": np.nan, "not mentioned": np.nan, "unmentioned": np.nan,
    }

    return (
        s.astype(str)
        .str.strip()
        .str.lower()
        .map(mapping)
        .astype(float)
    )

sources = {}
source_meta = []

for _, row in discovery.iterrows():
    p = row["path"]
    raw = safe_read_table(p)
    if raw is None:
        continue

    uid_col = infer_uid_col(raw)
    label_cols = infer_label_cols(raw)
    if uid_col is None or not label_cols:
        continue

    norm = pd.DataFrame({
        "StudyInstanceUID": raw[uid_col].astype(str)
    })

    mapped = []
    for label, col in label_cols.items():
        norm[label] = coerce_score(raw[col])
        mapped.append(label)

    # Aggregate duplicate StudyInstanceUID rows conservatively by mean
    norm = norm.groupby("StudyInstanceUID", as_index=False).mean(numeric_only=True)

    key = f"S{len(sources)+1:02d}"
    sources[key] = norm

    vals = norm[[c for c in LABELS if c in norm.columns]].stack().dropna()
    binary_fraction = float(vals.isin([0.0, 1.0]).mean()) if len(vals) else np.nan

    source_meta.append({
        "source_id": key,
        "source_name": row["source_name"],
        "path": p,
        "n_rows_normalized": len(norm),
        "n_labels": len(mapped),
        "mapped_labels": ", ".join(mapped),
        "binary_fraction": binary_fraction,
        "min_score": float(vals.min()) if len(vals) else np.nan,
        "max_score": float(vals.max()) if len(vals) else np.nan,
    })

source_manifest = pd.DataFrame(source_meta)

print("Normalized sources:", len(source_manifest))
display(source_manifest)



## 7. Gold evaluation

Vì chỉ có 58 gold studies:
- `ROC-AUC` là metric gần với competition nhất
- nhưng uncertainty rất lớn, nhất là pathology có rất ít positive cases
- notebook thêm bootstrap CI để thấy độ bất định

Không nên chọn label source chỉ vì hơn `0.01` trên 58 gold nếu CI chồng lấp mạnh.


In [ ]:

# ============================================================
# 7. Metrics helpers
# ============================================================
def safe_auc(y, p):
    mask = np.isfinite(y) & np.isfinite(p)
    y = np.asarray(y)[mask]
    p = np.asarray(p)[mask]

    if len(y) < 3 or len(np.unique(y)) < 2:
        return np.nan
    return roc_auc_score(y, p)

def safe_ap(y, p):
    mask = np.isfinite(y) & np.isfinite(p)
    y = np.asarray(y)[mask]
    p = np.asarray(p)[mask]

    if len(y) < 3 or np.sum(y == 1) == 0:
        return np.nan
    return average_precision_score(y, p)

def bootstrap_auc_ci(y, p, n_iter=500, seed=42):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    mask = np.isfinite(y) & np.isfinite(p)
    y, p = y[mask], p[mask]

    if len(y) < 3 or len(np.unique(y)) < 2:
        return np.nan, np.nan

    rng = np.random.default_rng(seed)
    vals = []

    for _ in range(n_iter):
        idx = rng.integers(0, len(y), len(y))
        yy, pp = y[idx], p[idx]
        if len(np.unique(yy)) < 2:
            continue
        vals.append(roc_auc_score(yy, pp))

    if len(vals) < 20:
        return np.nan, np.nan

    return (
        float(np.percentile(vals, 2.5)),
        float(np.percentile(vals, 97.5))
    )


In [ ]:

# ============================================================
# 8. Evaluate every source vs GOLD, per target
# ============================================================
metric_rows = []

for source_id, sdf in sources.items():
    merged = gold.copy()
    merged["StudyInstanceUID"] = merged["StudyInstanceUID"].astype(str)
    merged = merged.merge(sdf, on="StudyInstanceUID", how="left", suffixes=("_gold", "_pred"))

    for label in LABELS:
        pred_col = label + "_pred"
        gold_col = label + "_gold"

        if pred_col not in merged.columns:
            continue

        y = pd.to_numeric(merged[gold_col], errors="coerce").to_numpy(float)
        p = pd.to_numeric(merged[pred_col], errors="coerce").to_numpy(float)

        mask = np.isfinite(y) & np.isfinite(p)
        yy, pp = y[mask], p[mask]

        if len(yy) == 0:
            continue

        auc = safe_auc(yy, pp)
        ap = safe_ap(yy, pp)
        lo, hi = bootstrap_auc_ci(yy, pp, BOOTSTRAP_ITERS, SEED)

        pred_hard = (pp >= 0.5).astype(int)

        metric_rows.append({
            "source_id": source_id,
            "label": label,
            "n_gold_overlap": int(len(yy)),
            "n_positive": int(np.sum(yy == 1)),
            "n_negative": int(np.sum(yy == 0)),
            "coverage": float(len(yy) / len(gold)),
            "auc": auc,
            "auc_ci_low": lo,
            "auc_ci_high": hi,
            "ap": ap,
            "accuracy_05": accuracy_score(yy, pred_hard) if len(yy) else np.nan,
            "balanced_accuracy_05": balanced_accuracy_score(yy, pred_hard)
                if len(np.unique(yy)) == 2 else np.nan,
            "precision_05": precision_score(yy, pred_hard, zero_division=0),
            "recall_05": recall_score(yy, pred_hard, zero_division=0),
            "f1_05": f1_score(yy, pred_hard, zero_division=0),
        })

metrics = pd.DataFrame(metric_rows)

print("Per-label gold audit:")
display(metrics.sort_values(["label", "auc"], ascending=[True, False]))


In [ ]:

# ============================================================
# 9. Source-level macro summary
# ============================================================
if len(metrics):
    macro = (
        metrics.groupby("source_id")
        .agg(
            macro_auc=("auc", "mean"),
            macro_ap=("ap", "mean"),
            mean_coverage=("coverage", "mean"),
            labels_evaluated=("label", "nunique"),
        )
        .sort_values("macro_auc", ascending=False)
        .reset_index()
        .merge(source_manifest, on="source_id", how="left")
    )

    display(macro)

    fig, ax = plt.subplots(figsize=(9, 4))
    plot_df = macro.sort_values("macro_auc")
    ax.barh(plot_df["source_id"], plot_df["macro_auc"])
    ax.set_xlabel("Mean per-label ROC-AUC on 58 gold studies")
    ax.set_title("Pseudo-label source audit")
    plt.tight_layout()
    plt.show()
else:
    macro = pd.DataFrame()
    print("No source metrics available.")


In [ ]:

# ============================================================
# 10. Per-target source comparison
# ============================================================
if len(metrics):
    pivot_auc = metrics.pivot(index="label", columns="source_id", values="auc")
    print("Per-target AUC:")
    display(pivot_auc.round(3))

    fig, ax = plt.subplots(figsize=(max(7, len(sources)*1.2), 7))
    mat = pivot_auc.reindex(LABELS).to_numpy(dtype=float)
    im = ax.imshow(mat, aspect="auto", vmin=0.0, vmax=1.0)

    ax.set_yticks(range(len(LABELS)))
    ax.set_yticklabels(LABELS)
    ax.set_xticks(range(len(pivot_auc.columns)))
    ax.set_xticklabels(pivot_auc.columns, rotation=45, ha="right")
    ax.set_title("Gold ROC-AUC by pathology and pseudo-label source")
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


In [ ]:

# ============================================================
# 11. Pairwise disagreement across ALL competition studies
# ============================================================
pairwise_rows = []

source_ids = list(sources.keys())

for i in range(len(source_ids)):
    for j in range(i+1, len(source_ids)):
        a_id, b_id = source_ids[i], source_ids[j]
        a, b = sources[a_id], sources[b_id]

        m = a.merge(b, on="StudyInstanceUID", how="inner", suffixes=("_a", "_b"))

        for label in LABELS:
            ca, cb = label + "_a", label + "_b"
            if ca not in m.columns or cb not in m.columns:
                continue

            x = pd.to_numeric(m[ca], errors="coerce").to_numpy(float)
            y = pd.to_numeric(m[cb], errors="coerce").to_numpy(float)
            mask = np.isfinite(x) & np.isfinite(y)
            x, y = x[mask], y[mask]

            if len(x) == 0:
                continue

            hard_disagree = np.mean((x >= .5) != (y >= .5))
            mad = np.mean(np.abs(x - y))

            pairwise_rows.append({
                "source_a": a_id,
                "source_b": b_id,
                "label": label,
                "n_overlap": int(len(x)),
                "hard_disagreement_rate": float(hard_disagree),
                "mean_abs_score_difference": float(mad),
            })

pairwise = pd.DataFrame(pairwise_rows)

print("Pairwise disagreement:")
display(
    pairwise.sort_values(
        ["hard_disagreement_rate", "mean_abs_score_difference"],
        ascending=False
    ).head(80)
)


In [ ]:

# ============================================================
# 12. Build all-source prediction cube for 4,407 studies
# ============================================================
uids = train[["StudyInstanceUID"]].copy()
uids["StudyInstanceUID"] = uids["StudyInstanceUID"].astype(str)

cube = uids.copy()

for source_id, sdf in sources.items():
    tmp = sdf.copy()
    tmp["StudyInstanceUID"] = tmp["StudyInstanceUID"].astype(str)
    rename = {
        label: f"{source_id}__{label}"
        for label in LABELS if label in tmp.columns
    }
    tmp = tmp.rename(columns=rename)
    cube = cube.merge(tmp, on="StudyInstanceUID", how="left")

print("Prediction cube shape:", cube.shape)
display(cube.head())


In [ ]:

# ============================================================
# 13. Candidate silver labels
#
# We create FOUR candidates:
# A) mean consensus
# B) median consensus
# C) best source per target according to gold AUC
# D) gold-AUC weighted consensus (EXPERIMENTAL)
#
# IMPORTANT:
# 58 gold studies are removed from exported silver training pool.
# ============================================================
silver_mean = uids.copy()
silver_median = uids.copy()
silver_best = uids.copy()
silver_weighted = uids.copy()

best_source_per_target = {}
weight_table = []

for label in LABELS:
    cols = [f"{sid}__{label}" for sid in source_ids if f"{sid}__{label}" in cube.columns]

    if not cols:
        silver_mean[label] = np.nan
        silver_median[label] = np.nan
        silver_best[label] = np.nan
        silver_weighted[label] = np.nan
        continue

    arr = cube[cols].to_numpy(dtype=float)
    silver_mean[label] = np.nanmean(arr, axis=1)
    silver_median[label] = np.nanmedian(arr, axis=1)

    # Best source by gold AUC; tie -> better coverage
    mm = metrics[metrics["label"] == label].copy()
    if len(mm):
        mm = mm.sort_values(["auc", "coverage"], ascending=False)
        best_sid = mm.iloc[0]["source_id"]
        best_source_per_target[label] = best_sid
        best_col = f"{best_sid}__{label}"
        silver_best[label] = cube[best_col] if best_col in cube.columns else np.nan
    else:
        best_sid = None
        silver_best[label] = silver_mean[label]

    # Experimental reliability weighting:
    # use positive evidence above random, but never let one tiny gold set dominate
    weights = []
    weighted_cols = []

    for sid in source_ids:
        col = f"{sid}__{label}"
        if col not in cube.columns:
            continue

        r = metrics[(metrics["source_id"] == sid) & (metrics["label"] == label)]
        if len(r) == 0 or not np.isfinite(r.iloc[0]["auc"]):
            w = 0.05
        else:
            auc = float(r.iloc[0]["auc"])
            coverage = float(r.iloc[0]["coverage"])
            w = max(auc - 0.5, 0.01) * max(coverage, 0.25)

        weights.append(w)
        weighted_cols.append(col)
        weight_table.append({
            "label": label,
            "source_id": sid,
            "gold_weight_experimental": w
        })

    X = cube[weighted_cols].to_numpy(dtype=float)
    W = np.asarray(weights, dtype=float)

    out = []
    for row in X:
        valid = np.isfinite(row)
        if not valid.any():
            out.append(np.nan)
        else:
            ww = W[valid]
            xx = row[valid]
            if ww.sum() <= 0:
                out.append(float(np.mean(xx)))
            else:
                out.append(float(np.sum(xx * ww) / np.sum(ww)))

    silver_weighted[label] = out

weight_table = pd.DataFrame(weight_table)

print("Best source per target:")
for k, v in best_source_per_target.items():
    print(f"{k:18s} -> {v}")


In [ ]:

# ============================================================
# 14. Keep only NON-GOLD studies in silver exports
# ============================================================
gold_ids = set(gold["StudyInstanceUID"].astype(str))

def drop_gold(df):
    return df[~df["StudyInstanceUID"].astype(str).isin(gold_ids)].reset_index(drop=True)

silver_mean_weak = drop_gold(silver_mean)
silver_median_weak = drop_gold(silver_median)
silver_best_weak = drop_gold(silver_best)
silver_weighted_weak = drop_gold(silver_weighted)

print("Silver weak-pool sizes:")
print("mean:", len(silver_mean_weak))
print("median:", len(silver_median_weak))
print("best:", len(silver_best_weak))
print("weighted:", len(silver_weighted_weak))


In [ ]:

# ============================================================
# 15. Find high-disagreement studies for manual audit
# ============================================================
disagreement_cases = []

for label in LABELS:
    cols = [f"{sid}__{label}" for sid in source_ids if f"{sid}__{label}" in cube.columns]
    if len(cols) < 2:
        continue

    vals = cube[cols]
    spread = vals.max(axis=1, skipna=True) - vals.min(axis=1, skipna=True)

    tmp = pd.DataFrame({
        "StudyInstanceUID": cube["StudyInstanceUID"],
        "label": label,
        "score_spread": spread,
        "n_sources": vals.notna().sum(axis=1),
    })

    tmp = tmp[tmp["n_sources"] >= 2]
    disagreement_cases.append(tmp)

if disagreement_cases:
    disagreement_cases = pd.concat(disagreement_cases, ignore_index=True)
    disagreement_cases = disagreement_cases.sort_values(
        ["score_spread", "n_sources"], ascending=False
    )

    # Add report for audit convenience
    disagreement_cases = disagreement_cases.merge(
        train[["StudyInstanceUID", "Report"]],
        on="StudyInstanceUID",
        how="left"
    )

    display(disagreement_cases.head(100))
else:
    disagreement_cases = pd.DataFrame()



# 16. External Data Research Manifest

Đây là **manifest nghiên cứu**, không phải phán quyết pháp lý.

Mỗi dataset phải qua 4 cổng trước khi dùng:

1. **Competition legality / license**
2. **MRI protocol similarity**
3. **Pathology overlap**
4. **Controlled experiment gain**

Trạng thái:
- `VERIFY`: cần xác nhận terms trước khi dùng
- `HOLD`: chưa ưu tiên tải/train
- `CANDIDATE`: đáng chuẩn bị experiment
- `REJECT`: không phù hợp / license không ổn


In [ ]:

# ============================================================
# 16. External data manifest
# ============================================================
external_manifest = pd.DataFrame([
    {
        "dataset": "MRNet",
        "main_targets": "ACL; Medial Meniscus; Lateral Meniscus; general abnormality",
        "modality_protocol_match": "High: clinical multi-plane knee MRI",
        "best_use": "knee-MRI pretraining / ACL-meniscus specialist",
        "localization_labels": "No strong dense localization as main feature",
        "license_status": "VERIFY",
        "priority": "HIGH",
        "notes": "Scientifically strong match; do not use until competition-use terms are cleared."
    },
    {
        "dataset": "MOST",
        "main_targets": "Medial OA; Lateral OA; PF OA; cartilage; meniscus",
        "modality_protocol_match": "Moderate-High for degenerative knee MRI",
        "best_use": "OA specialist / structural degeneration pretraining",
        "localization_labels": "Semiquantitative structural readings",
        "license_status": "VERIFY",
        "priority": "HIGH",
        "notes": "Potentially valuable for OA heads; access/DUA needs audit."
    },
    {
        "dataset": "OAI",
        "main_targets": "Medial OA; Lateral OA; PF OA; cartilage; meniscus",
        "modality_protocol_match": "High for OA MRI, less direct for acute injury",
        "best_use": "OA representation / cartilage-meniscus specialist",
        "localization_labels": "Rich structural/imaging-derived OA information",
        "license_status": "VERIFY",
        "priority": "HIGH",
        "notes": "Large and longitudinal; not first choice for ACL/MCL/fracture."
    },
    {
        "dataset": "SKM-TEA",
        "main_targets": "Meniscus; cartilage; anatomy; pathology localization",
        "modality_protocol_match": "Moderate: qDESS differs from competition",
        "best_use": "anatomy localizer / ROI pretraining",
        "localization_labels": "Segmentation + pathology boxes",
        "license_status": "VERIFY",
        "priority": "HIGH",
        "notes": "More useful for learning WHERE than direct classification."
    },
    {
        "dataset": "fastMRI+",
        "main_targets": "Broad pathology localization; meniscus/cartilage/bone findings",
        "modality_protocol_match": "Moderate",
        "best_use": "pathology localization / attention pretraining",
        "localization_labels": "Bounding boxes + study labels",
        "license_status": "VERIFY",
        "priority": "MEDIUM",
        "notes": "Very attractive scientifically; license compatibility must be confirmed."
    },
    {
        "dataset": "KneeMRI ACL (Croatia)",
        "main_targets": "ACL",
        "modality_protocol_match": "Moderate-High for sagittal PD-FS ACL",
        "best_use": "ACL specialist / ACL ROI representation",
        "localization_labels": "ACL-focused ROI/labels",
        "license_status": "VERIFY",
        "priority": "MEDIUM",
        "notes": "Narrow but potentially useful for ACL."
    },
    {
        "dataset": "RadImageNet pretrained weights",
        "main_targets": "Generic medical imaging representation",
        "modality_protocol_match": "Generic medical domain",
        "best_use": "medical-pretrained image encoder",
        "localization_labels": "N/A",
        "license_status": "VERIFY MODEL LICENSE",
        "priority": "HIGH",
        "notes": "Already appears in strong public RSNA pipelines."
    },
    {
        "dataset": "DINOv2 pretrained weights",
        "main_targets": "Generic visual representation",
        "modality_protocol_match": "Generic natural-image SSL",
        "best_use": "baseline encoder",
        "localization_labels": "N/A",
        "license_status": "VERIFY MODEL LICENSE",
        "priority": "HIGH",
        "notes": "Strong public baseline backbone; small first, bigger only as controlled experiment."
    },
])

display(external_manifest)



# 17. Knee MRI Domain-Knowledge Map

`evidence_level` ở đây chỉ dùng để tổ chức experiment:

- `STRONG_CLINICAL`: mối liên hệ plane/sequence/anatomy là kiến thức MRI lâm sàng ổn định
- `PLAUSIBLE_MODELING`: ý tưởng modeling hợp lý nhưng vẫn phải ablate
- Không có dòng nào có nghĩa là “chắc chắn tăng Kaggle score”.

Điểm quan trọng: **không hard-code pixel weights trước**.  
Domain knowledge nên được biến thành:
- sequence routing
- task-specific attention
- ROI/localizer branch
- global + local fusion
- multi-resolution modeling


In [ ]:

# ============================================================
# 17. Pathology × anatomy × plane × sequence × modeling map
# ============================================================
domain_map = pd.DataFrame([
    {
        "target": "ACL",
        "primary_anatomy": "Anterior cruciate ligament / intercondylar notch",
        "preferred_planes": "Sagittal primary; coronal/axial supportive",
        "preferred_sequences": "PD/T2 fluid-sensitive, often fat-suppressed",
        "secondary_signs": "Bone contusion pattern, fiber discontinuity, orientation change",
        "modeling_hook": "task-specific attention; optional ACL local ROI + global branch",
        "evidence_level": "STRONG_CLINICAL + PLAUSIBLE_MODELING"
    },
    {
        "target": "MCL",
        "primary_anatomy": "Medial collateral ligament / medial knee",
        "preferred_planes": "Coronal primary; axial supportive",
        "preferred_sequences": "Fluid-sensitive fat-suppressed",
        "secondary_signs": "Periligamentous edema, fiber disruption",
        "modeling_hook": "medial-side ROI/localizer; task-specific plane weighting",
        "evidence_level": "STRONG_CLINICAL + PLAUSIBLE_MODELING"
    },
    {
        "target": "Medial Meniscus",
        "primary_anatomy": "Medial meniscus anterior horn/body/posterior horn",
        "preferred_planes": "Sagittal + coronal",
        "preferred_sequences": "PD/intermediate-weighted ± fat suppression",
        "secondary_signs": "Surface-reaching signal, displaced fragments, extrusion",
        "modeling_hook": "high-resolution joint-line ROI; multi-plane local branch",
        "evidence_level": "STRONG_CLINICAL + PLAUSIBLE_MODELING"
    },
    {
        "target": "Lateral Meniscus",
        "primary_anatomy": "Lateral meniscus anterior horn/body/posterior horn",
        "preferred_planes": "Sagittal + coronal; axial sometimes supportive",
        "preferred_sequences": "PD/intermediate-weighted ± fat suppression",
        "secondary_signs": "Surface-reaching signal, morphology change, displaced fragment",
        "modeling_hook": "high-resolution lateral joint-line ROI",
        "evidence_level": "STRONG_CLINICAL + PLAUSIBLE_MODELING"
    },
    {
        "target": "Medial OA",
        "primary_anatomy": "Medial tibiofemoral cartilage/subchondral bone/meniscus",
        "preferred_planes": "Coronal + sagittal",
        "preferred_sequences": "Morphologic non-fluid + fluid-sensitive complement",
        "secondary_signs": "Cartilage loss, marrow lesions, meniscal extrusion, osteophytes",
        "modeling_hook": "medial compartment ROI + global structural context",
        "evidence_level": "STRONG_CLINICAL + PLAUSIBLE_MODELING"
    },
    {
        "target": "Lateral OA",
        "primary_anatomy": "Lateral tibiofemoral cartilage/subchondral bone/meniscus",
        "preferred_planes": "Coronal + sagittal",
        "preferred_sequences": "Morphologic non-fluid + fluid-sensitive complement",
        "secondary_signs": "Cartilage loss, marrow lesions, meniscal extrusion, osteophytes",
        "modeling_hook": "lateral compartment ROI + global structural context",
        "evidence_level": "STRONG_CLINICAL + PLAUSIBLE_MODELING"
    },
    {
        "target": "PF OA",
        "primary_anatomy": "Patella / trochlea / patellofemoral cartilage",
        "preferred_planes": "Axial primary; sagittal supportive",
        "preferred_sequences": "Morphologic + fluid-sensitive",
        "secondary_signs": "Cartilage loss, subchondral change, osteophytes",
        "modeling_hook": "axial PF ROI + sagittal context",
        "evidence_level": "STRONG_CLINICAL + PLAUSIBLE_MODELING"
    },
    {
        "target": "Effusion",
        "primary_anatomy": "Suprapatellar pouch and joint recesses",
        "preferred_planes": "Sagittal/axial/coronal all informative",
        "preferred_sequences": "Fluid-sensitive",
        "secondary_signs": "Distention of joint recesses",
        "modeling_hook": "global-fluid branch; avoid overly tight ROI",
        "evidence_level": "STRONG_CLINICAL + PLAUSIBLE_MODELING"
    },
    {
        "target": "Synovitis",
        "primary_anatomy": "Synovium / Hoffa fat pad / joint recesses",
        "preferred_planes": "Multi-plane",
        "preferred_sequences": "Fluid-sensitive; non-contrast ambiguity remains",
        "secondary_signs": "Synovial thickening/signal, inflammatory context",
        "modeling_hook": "medical-pretrained encoder; broader attention; do not over-localize",
        "evidence_level": "STRONG_CLINICAL + PLAUSIBLE_MODELING"
    },
    {
        "target": "Baker's",
        "primary_anatomy": "Posteromedial popliteal region / gastrocnemius-semimembranosus bursa",
        "preferred_planes": "Axial + sagittal",
        "preferred_sequences": "T2/PD fluid-sensitive",
        "secondary_signs": "Fluid-filled posteromedial cystic structure",
        "modeling_hook": "posteromedial ROI candidate + global branch",
        "evidence_level": "STRONG_CLINICAL + PLAUSIBLE_MODELING"
    },
    {
        "target": "Contusion",
        "primary_anatomy": "Subchondral bone marrow across femur/tibia/patella",
        "preferred_planes": "Multi-plane",
        "preferred_sequences": "Fluid-sensitive fat-suppressed / STIR-like",
        "secondary_signs": "Marrow edema pattern; may support ligament injury",
        "modeling_hook": "global bone-aware attention; keep wide FOV",
        "evidence_level": "STRONG_CLINICAL + PLAUSIBLE_MODELING"
    },
    {
        "target": "Fracture",
        "primary_anatomy": "Cortex/subchondral bone / marrow",
        "preferred_planes": "Multi-plane",
        "preferred_sequences": "T1/non-fluid + fluid-sensitive complement",
        "secondary_signs": "Low-signal fracture line + marrow edema",
        "modeling_hook": "multi-sequence global/local fusion",
        "evidence_level": "STRONG_CLINICAL + PLAUSIBLE_MODELING"
    },
])

display(domain_map)



# 18. Baseline direction implied by EDA + label audit + domain knowledge

Sau notebook này, **chưa tự động train**.

Baseline V1 dự kiến chỉ được khóa sau khi ta xem artifact:

```text
REPORT SIDE
    2–N pseudo-label sources
            ↓
    gold audit per target
            ↓
    chosen silver labels
            ↓
────────────────────────────────
IMAGE SIDE
    correct DICOM geometry sorting
            ↓
    physical-space crop (~130–140 mm as first candidate)
            ↓
    multi-plane / fluid vs non-fluid routing
            ↓
    2.5D slice groups
            ↓
    compact encoder first
            ↓
    task-specific slice/sequence attention
            ↓
    12 outputs
────────────────────────────────
VALIDATION
    report-group-safe CV
    + untouched 58 gold audit
```

Controlled experiments sau baseline sẽ thay **một yếu tố/lần**:
- silver-label strategy
- image resolution
- physical FOV
- slice sampling
- normalization
- attention pooling
- task-specific attention
- ROI/local branches
- RadImageNet vs DINO
- external knee MRI pretraining
- specialist models
- ensemble cuối cùng


In [ ]:

# ============================================================
# 19. Export artifacts
# ============================================================
OUT = Path("/kaggle/working/rsna_label_audit_v1")
if not Path("/kaggle/working").exists():
    OUT = Path("/mnt/data/rsna_label_audit_v1")
OUT.mkdir(parents=True, exist_ok=True)

# Core artifacts
source_manifest.to_csv(OUT / "pseudo_label_source_manifest.csv", index=False)
metrics.to_csv(OUT / "gold_per_label_metrics.csv", index=False)
macro.to_csv(OUT / "gold_macro_source_summary.csv", index=False)
pairwise.to_csv(OUT / "pairwise_source_disagreement.csv", index=False)
weight_table.to_csv(OUT / "experimental_gold_weights.csv", index=False)

external_manifest.to_csv(OUT / "external_data_manifest.csv", index=False)
domain_map.to_csv(OUT / "knee_domain_knowledge_map.csv", index=False)

if len(disagreement_cases):
    disagreement_cases.head(1000).to_csv(
        OUT / "top_disagreement_cases.csv", index=False
    )

silver_mean_weak.to_csv(OUT / "silver_mean_consensus.csv", index=False)
silver_median_weak.to_csv(OUT / "silver_median_consensus.csv", index=False)
silver_best_weak.to_csv(OUT / "silver_best_source_per_target.csv", index=False)
silver_weighted_weak.to_csv(
    OUT / "silver_gold_weighted_EXPERIMENTAL.csv", index=False
)

# Human-readable result summary
lines = []
lines.append("RSNA KNEE LABEL AUDIT v1 — RESULTS TO SEND BACK")
lines.append("=" * 72)
lines.append(f"Competition root: {COMP_ROOT}")
lines.append(f"Gold studies: {len(gold)}")
lines.append(f"Pseudo-label sources found: {len(source_manifest)}")
lines.append("")

if len(source_manifest):
    lines.append("SOURCES")
    for _, r in source_manifest.iterrows():
        lines.append(
            f"- {r['source_id']}: {r['source_name']} | "
            f"labels={r['n_labels']} | binary_fraction={r['binary_fraction']:.3f}"
        )
    lines.append("")

if len(macro):
    lines.append("MACRO GOLD AUDIT")
    for _, r in macro.iterrows():
        lines.append(
            f"- {r['source_id']}: macro_auc={r['macro_auc']:.4f} | "
            f"macro_ap={r['macro_ap']:.4f} | "
            f"coverage={r['mean_coverage']:.4f}"
        )
    lines.append("")

if len(metrics):
    lines.append("BEST SOURCE PER TARGET BY GOLD AUC")
    for label in LABELS:
        mm = metrics[metrics["label"] == label].sort_values(
            ["auc", "coverage"], ascending=False
        )
        if len(mm):
            r = mm.iloc[0]
            lines.append(
                f"- {label}: {r['source_id']} | AUC={r['auc']:.4f} "
                f"[{r['auc_ci_low']:.4f}, {r['auc_ci_high']:.4f}] "
                f"| n={int(r['n_gold_overlap'])}"
            )
    lines.append("")

lines.append("EXTERNAL DATA PRIORITY")
for _, r in external_manifest.iterrows():
    lines.append(
        f"- {r['dataset']}: priority={r['priority']} | "
        f"status={r['license_status']} | use={r['best_use']}"
    )

lines.append("")
lines.append("NEXT:")
lines.append("Send this TXT or the ZIP back to ChatGPT.")
lines.append("Do NOT choose a final silver-label source only from macro score.")
lines.append("We will inspect per-target reliability + uncertainty first.")

result_text = "\n".join(lines)
(OUT / "RESULTS_TO_SEND_BACK.txt").write_text(result_text, encoding="utf-8")

print(result_text)

ZIP_PATH = OUT.parent / "RSNA_Knee_Label_Audit_v1_Artifacts.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p, arcname=p.relative_to(OUT))

print("\nArtifacts:", OUT)
print("ZIP:", ZIP_PATH)
